In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent / 'src'))

from kafka import KafkaConsumer, TopicPartition
from models import Ride, ride_deserializer
import psycopg2
from datetime import datetime
import time

server = 'localhost:9092'
topic_name = 'rides'

In [2]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [ ]:
conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True

cur = conn.cursor()

In [ ]:
print(f"Listening to {topic_name} and writing to PostgreSQL")

count = 0
t0 = time.time()

consumer.subscribe(['rides'])

while not consumer.assignment():
    consumer.poll(timeout_ms=100)

end_offsets = consumer.end_offsets(consumer.assignment())

for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute("""
            INSERT INTO processed_events
                (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
                VALUES (%s, %s, %s, %s, %s)""",
                (ride.PULocationID, ride.DOLocationID,
                 ride.trip_distance, ride.total_amount, pickup_dt)
                )
    count += 1
    if count % 1000 == 0:
        print(f"Inserted {count} rows")
        conn.commit()

    tp = TopicPartition(message.topic, message.partition)
    if consumer.position(tp) >= end_offsets[tp]:
        break

conn.commit()
t1 = time.time()

print(f"Took {t1 - t0} seconds to insert {count} rows")

consumer.close()
cur.close()
conn.close()